# Validação das faixas de confiança — visão de desenvolvimento (ticket #9)

**Dados:** somente a visão de desenvolvimento (via `carregar_serie` e `carregar_particoes`). O período OOT não é lido por nenhuma célula (ADR 0002).

**Critérios fixados antes de qualquer resultado:** `docs/relatorio/fase2/9-pre-registro.md` (SHA-256 `0e0c3e73…fd84505`, gravado às 2026-10-30T05:00 -03, antes do primeiro MASE por fold).

**Status:** Observações e Interpretações abaixo são **PROPOSTAS** e aguardam aprovação do usuário. Nada aqui seleciona modelo: a confiança é metadado da versão (ADR 0008) e os 20 runs são só baselines.

**Limite:** a validação só vê baselines e precisa ser repetida depois de integrar #10 e #11.

## Perguntas
1. Qual o MASE, a dispersão entre folds e a faixa de cada baseline nas 8 séries?
2. As faixas discriminam algo nesses dados (condições D1 a D3 do pré-registro)?
3. O critério de instabilidade é sensível ao nível do IC, à saída de um fold e à semana 07-19 do item C?
4. Trocar a escala do MASE na diária (lag 1 para lag 7) muda o quadro (condições E1 e E2)?

In [1]:
import os
import warnings

os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'
warnings.filterwarnings('ignore')

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from case_mr_health import algoritmos, confianca
from case_mr_health.backtest import PERIODOS_ATIPICOS, rodar_backtest
from case_mr_health.candidatos import SKUS, candidatos
from case_mr_health.config import Settings
from case_mr_health.particoes import GRANULARIDADES, carregar_particoes
from case_mr_health.treino import aplicar_seed, carregar_serie

SEED = 20261030
FRONTEIRAS_INSTAVEL = 2
MIN_FAIXAS = 2
MAX_RUNS_STATUS = 16
MIN_RUNS_CLASSIFICADOS = 10
RAZAO_MIN, RAZAO_MAX = 0.90, 1.10
ORDEM = (*confianca.FAIXAS, confianca.INSUFICIENTE)
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'pyproject.toml').is_file()
)
os.chdir(ROOT)  # caminhos da configuração são relativos à raiz
cfg = Settings()  # só a visão de desenvolvimento é aberta
LIMITES = cfg.confianca_limites
aplicar_seed(SEED)
pd.set_option('display.width', 160, 'display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

/home/rwallan/git/case-mr-health/.claude/worktrees/agent-aa5a7379f6dbeaad4/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## 1. MASE, dispersão e faixa por série e baseline

MASE por fold com a escala aprovada em D-006 (naive de um passo no treino do fold). `ep = dp/√n`; IC t de 80% truncado em 0; `fronteiras` é quantas fronteiras de faixa o IC atravessa. Instável: duas ou mais.

In [3]:
particoes = {(p.sku, p.granularidade): p for p in carregar_particoes(cfg)}
series, backtests = {}, {}
for sku in SKUS:
    for gran in GRANULARIDADES:
        series[sku, gran] = carregar_serie(cfg, sku, gran)
        for c in candidatos(sku, gran):
            if not c.entra or c.algoritmo not in algoritmos.ALGORITMOS:
                continue
            for hp in c.combinacoes():
                aplicar_seed(SEED)
                backtests[sku, gran, c.algoritmo] = rodar_backtest(
                    series[sku, gran],
                    particoes[sku, gran],
                    c.algoritmo,
                    hp,
                    SEED,
                )


def mases(chave):
    return [r.metricas['mase'] for r in backtests[chave].folds]


def linha(chave, valores=None):
    valores = mases(chave) if valores is None else valores
    c = confianca.avaliar(valores, cfg)
    n = len(valores)
    superior = confianca.indice_faixa(c.ic_superior, LIMITES)
    fronteiras = superior - confianca.indice_faixa(c.ic_inferior, LIMITES)
    return {
        'sku': chave[0].removeprefix('item '),
        'gran': chave[1],
        'baseline': chave[2],
        'n': n,
        'mase': c.mase,
        'dp': c.dp,
        'ep': c.dp / np.sqrt(n),
        'ic80_inf': c.ic_inferior,
        'ic80_sup': c.ic_superior,
        'fronteiras': fronteiras,
        'pontual': c.faixa_pontual,
        'status': c.status,
    }


tabela = pd.DataFrame([linha(k) for k in backtests])
print(len(tabela), 'runs')
tabela

20 runs


,sku,gran,baseline,n,mase,dp,ep,ic80_inf,ic80_sup,fronteiras,pontual,status
0,A,diaria,naive,5,0.892,0.414,0.185,0.608,1.176,2,Média,Insuficiente
1,A,diaria,media_historica,5,0.849,0.248,0.111,0.679,1.019,2,Média,Insuficiente
2,A,diaria,naive_sazonal,5,0.811,0.218,0.098,0.661,0.960,1,Média,Média
3,A,semanal,naive,4,1.118,0.837,0.419,0.432,1.803,3,Ruim,Insuficiente
4,A,semanal,media_historica,4,0.664,0.595,0.297,0.177,1.151,3,Boa,Insuficiente
5,B,diaria,naive,4,0.744,0.358,0.179,0.451,1.037,3,Boa,Insuficiente
6,B,diaria,media_historica,4,0.879,0.252,0.126,0.673,1.086,2,Média,Insuficiente
7,B,diaria,naive_sazonal,4,0.877,0.240,0.120,0.681,1.074,2,Média,Insuficiente
8,B,semanal,naive,4,0.868,0.527,0.263,0.437,1.300,3,Média,Insuficiente
9,B,semanal,media_historica,4,0.605,0.385,0.192,0.290,0.920,2,Boa,Insuficiente


In [4]:
# Confere as tags gravadas pela CLI com o cálculo deste notebook. O MLflow só
# existe depois do `treinar` das 8 séries; sem ele, a célula apenas avisa.
import mlflow
from mlflow import MlflowClient

from case_mr_health.treino import nome_modelo

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
cliente = MlflowClient()
conferidos, divergentes = 0, []
for (sku, gran, alg), bt in backtests.items():
    versoes = [
        v
        for v in cliente.search_model_versions(
            f"name = '{nome_modelo(sku, gran)}'"
        )
        if v.tags.get('algoritmo') == alg
    ]
    if not versoes:
        continue
    conferidos += 1
    esperado = confianca.avaliar(mases((sku, gran, alg)), cfg).tags()
    ultima = max(versoes, key=lambda v: int(v.version))
    if {k: ultima.tags.get(k) for k in esperado} != esperado:
        divergentes.append((sku, gran, alg))
print('versões conferidas:', conferidos, '| divergentes:', divergentes)

versões conferidas: 20 | divergentes: []


**Observação (PROPOSTA).** Nos 20 runs o MASE vai de 0,605 (B semanal, média) a 2,459 (C semanal, naive). O dp entre folds vai de 0,127 a 0,414 nas diárias e de 0,385 a 2,959 nas semanais. Nenhum run tem faixa pontual Ótima. O naive, baseline trivial, fica com MASE de 0,744 a 0,892 nas diárias: faixa pontual Boa em B e Média em A, C e D. As tags gravadas pela CLI nas 20 versões conferem com este cálculo.

## 2. As faixas discriminam?

In [5]:
def contagem(coluna, dados):
    return (
        dados[coluna]
        .value_counts()
        .reindex(ORDEM, fill_value=0)
        .rename(coluna)
    )


distribuicao = pd.concat(
    [
        contagem('pontual', tabela).rename('pontual (20)'),
        contagem('status', tabela).rename('status (20)'),
        contagem('status', tabela[tabela.gran == 'diaria']).rename(
            'status diária (12)'
        ),
        contagem('status', tabela[tabela.gran == 'semanal']).rename(
            'status semanal (8)'
        ),
        contagem('pontual', tabela[tabela.gran == 'diaria']).rename(
            'pontual diária (12)'
        ),
        contagem('pontual', tabela[tabela.gran == 'semanal']).rename(
            'pontual semanal (8)'
        ),
    ],
    axis=1,
)


def discrimina(status):
    n = status.value_counts()
    faixas = [f for f in confianca.FAIXAS if n.get(f, 0) > 0]
    return {
        'D1 (>=2 faixas)': len(faixas) >= MIN_FAIXAS,
        'D2 (nenhum status com >=16)': int(n.max()) < MAX_RUNS_STATUS,
        'D3 (>=10 fora de Insuficiente)': int(
            len(status) - n.get(confianca.INSUFICIENTE, 0)
        )
        >= MIN_RUNS_CLASSIFICADOS,
        'faixas presentes': faixas,
    }


print(discrimina(tabela.status))
distribuicao

{'D1 (>=2 faixas)': True, 'D2 (nenhum status com >=16)': True, 'D3 (>=10 fora de Insuficiente)': False, 'faixas presentes': ['Média', 'Ruim']}


,pontual (20),status (20),status diária (12),status semanal (8),pontual diária (12),pontual semanal (8)
Ótima,0,0,0,0,0,0
Boa,4,0,0,0,2,2
Média,9,2,2,0,7,2
Ruim,7,3,3,0,3,4
Insuficiente,0,15,7,8,0,0


In [6]:
fronteiras = np.array(LIMITES)
tabela['dist_fronteira'] = [np.abs(fronteiras - m).min() for m in tabela.mase]
tabela['dist_em_ep'] = tabela.dist_fronteira / tabela.ep
print(
    'mediana dp por granularidade:',
    tabela.groupby('gran').dp.median().round(3).to_dict(),
)
print(
    'mediana ep por granularidade:',
    tabela.groupby('gran').ep.median().round(3).to_dict(),
)
print(
    'runs com distância à fronteira menor que 1 ep:',
    int((tabela.dist_em_ep < 1).sum()),
    'de',
    len(tabela),
)
tabela[
    ['sku', 'gran', 'baseline', 'mase', 'ep', 'dist_fronteira', 'dist_em_ep']
]

mediana dp por granularidade: {'diaria': 0.25, 'semanal': 0.883}
mediana ep por granularidade: {'diaria': 0.12, 'semanal': 0.442}
runs com distância à fronteira menor que 1 ep: 17 de 20


,sku,gran,baseline,mase,ep,dist_fronteira,dist_em_ep
0,A,diaria,naive,0.892,0.185,0.108,0.584
1,A,diaria,media_historica,0.849,0.111,0.099,0.894
2,A,diaria,naive_sazonal,0.811,0.098,0.061,0.623
3,A,semanal,naive,1.118,0.419,0.118,0.281
4,A,semanal,media_historica,0.664,0.297,0.086,0.289
5,B,diaria,naive,0.744,0.179,0.006,0.035
6,B,diaria,media_historica,0.879,0.126,0.121,0.959
7,B,diaria,naive_sazonal,0.877,0.120,0.123,1.022
8,B,semanal,naive,0.868,0.263,0.118,0.450
9,B,semanal,media_historica,0.605,0.192,0.105,0.545


**Observação (PROPOSTA).** Depois da regra de instabilidade: 15 Insuficiente, 3 Ruim, 2 Média. D1 e D2 passam, **D3 falha** (5 runs fora de Insuficiente, mínimo 10). Pelo critério pré-registrado, **as faixas não discriminam nestes dados**. As 8 execuções semanais são Insuficiente. A faixa pontual, sem a regra, distribui-se em 4 Boa, 9 Média e 7 Ruim, mas em 17 de 20 runs o MASE está a menos de 1 erro-padrão da fronteira mais próxima. Medianas do ep: 0,120 (diária) e 0,442 (semanal), contra faixas centrais de largura 0,25.

**Interpretação (PROPOSTA, hipótese).** O problema é de resolução, não de escala: o ep da média com 4 a 5 folds é da ordem de metade da largura de uma faixa na diária e quase o dobro na semanal, onde cada fold é um único ponto. A faixa pontual existe, mas o rótulo em geral não está determinado pelos dados. Isso descreve baselines; modelos melhores podem ter dispersão menor, ou não.

## 3. Sensibilidades pré-declaradas

In [7]:
def jackknife(valores):
    medias = [
        np.mean(valores[:i] + valores[i + 1 :]) for i in range(len(valores))
    ]
    indices = [confianca.indice_faixa(m, LIMITES) for m in medias]
    return min(medias), max(medias), max(indices) - min(indices)


linhas = []
for k in backtests:
    v = mases(k)
    _, _, inf95, sup95 = confianca.intervalo(v, 0.95)
    f95 = confianca.indice_faixa(sup95, LIMITES)
    f95 -= confianca.indice_faixa(inf95, LIMITES)
    jmin, jmax, fj = jackknife(v)
    linhas.append({
        'sku': k[0].removeprefix('item '),
        'gran': k[1],
        'baseline': k[2],
        'instavel_ic80': confianca.avaliar(v, cfg).motivo
        == confianca.MOTIVO_INSTAVEL,
        'ic95_inf': inf95,
        'ic95_sup': sup95,
        'instavel_ic95': f95 >= FRONTEIRAS_INSTAVEL,
        'jack_min': jmin,
        'jack_max': jmax,
        'instavel_jack': fj >= FRONTEIRAS_INSTAVEL,
    })
sens = pd.DataFrame(linhas)
colunas = ['instavel_ic80', 'instavel_ic95', 'instavel_jack']
print(sens[colunas].sum().to_dict())
sens

{'instavel_ic80': 15, 'instavel_ic95': 18, 'instavel_jack': 4}


,sku,gran,baseline,instavel_ic80,ic95_inf,ic95_sup,instavel_ic95,jack_min,jack_max,instavel_jack
0,A,diaria,naive,True,0.377,1.406,True,0.730,0.982,False
1,A,diaria,media_historica,True,0.542,1.156,True,0.768,0.939,False
2,A,diaria,naive_sazonal,False,0.540,1.082,True,0.748,0.870,False
3,A,semanal,naive,True,0.000,2.450,True,0.768,1.385,False
4,A,semanal,media_historica,True,0.000,1.611,True,0.478,0.877,True
5,B,diaria,naive,True,0.174,1.313,True,0.587,0.878,False
6,B,diaria,media_historica,True,0.478,1.280,True,0.763,0.955,False
7,B,diaria,naive_sazonal,True,0.496,1.259,True,0.783,0.965,False
8,B,semanal,naive,True,0.031,1.706,True,0.660,1.076,True
9,B,semanal,media_historica,True,0.000,1.217,True,0.429,0.725,False


In [8]:
linhas = []
for alg in ('naive', 'media_historica'):
    k = ('item C', 'semanal', alg)
    bt = backtests[k]
    atipico = PERIODOS_ATIPICOS['item C', 'semanal']
    todos = mases(k)
    sem_periodo = [
        r.metricas['mase'] for r in bt.folds if r.teste_inicio != atipico
    ]
    sem_folds = todos[2:]
    for nome, v in (
        ('todas', todos),
        ('sem semana 07-19', sem_periodo),
        ('sem folds 1 e 2', sem_folds),
    ):
        c = confianca.avaliar(v, cfg)
        linhas.append({
            'baseline': alg,
            'variante': nome,
            'n': len(v),
            'mase': c.mase,
            'dp': c.dp,
            'ic80_inf': c.ic_inferior,
            'ic80_sup': c.ic_superior,
            'pontual': c.faixa_pontual,
            'status': c.status,
        })
print(
    'MASE por fold, naive:',
    np.round(mases(('item C', 'semanal', 'naive')), 3),
)
print(
    'MASE por fold, média:',
    np.round(mases(('item C', 'semanal', 'media_historica')), 3),
)
pd.DataFrame(linhas)

MASE por fold, naive: [5.238 3.209 0.212 1.176]
MASE por fold, média: [6.27  0.06  0.172 1.007]


,baseline,variante,n,mase,dp,ic80_inf,ic80_sup,pontual,status
0,naive,todas,4,2.459,2.235,0.629,4.289,Ruim,Insuficiente
1,naive,sem semana 07-19,3,1.533,1.530,0.000,3.198,Ruim,Insuficiente
2,naive,sem folds 1 e 2,2,0.694,0.682,0.000,2.178,Boa,Insuficiente
3,media_historica,todas,4,1.877,2.959,0.000,4.300,Ruim,Insuficiente
4,media_historica,sem semana 07-19,3,0.413,0.517,0.000,0.976,Ótima,Insuficiente
5,media_historica,sem folds 1 e 2,2,0.589,0.590,0.000,1.873,Boa,Insuficiente


**Observação (PROPOSTA).** Instáveis: 15 de 20 com IC de 80%, 18 com IC de 95% e 4 com a saída de um fold (todos semanais). No item C semanal, a semana 07-19 é o fold 1 (MASE 5,238 no naive, 6,270 na média). Sem ela, a faixa pontual da média vai de Ruim (1,877) para Ótima (0,413), e o status segue Insuficiente nas três variantes dos dois baselines. Com 2 ou 3 pontos essas variantes têm IC muito largo.

**Interpretação (PROPOSTA, hipótese).** O resultado depende do critério: o IC mede a incerteza da média e o jackknife mede só a influência de um fold, que é uma pergunta menos exigente. A faixa pontual do item C semanal é ditada por uma semana, o que apoia não publicar rótulo pontual ali.

## 4. Alternativas, avaliadas **depois** de ver o resultado

Estas regras são **pós-hoc**. Servem para o usuário decidir; não substituem a pré-registrada. A: pré-registrada. B: tolera até 2 fronteiras. C: jackknife como critério (instável se a saída de um fold move o rótulo 2 ou mais faixas). D: três níveis, comparando o IC com 1.

In [9]:
def regra_ampliada(k):
    c = confianca.avaliar(
        mases(k), cfg.model_copy(update={'confianca_max_fronteiras_ic': 2})
    )
    return c.status


def regra_jackknife(k):
    v = mases(k)
    pontual = confianca.faixa(float(np.mean(v)), LIMITES)
    instavel = jackknife(v)[2] >= FRONTEIRAS_INSTAVEL
    return confianca.INSUFICIENTE if instavel else pontual


def regra_tres_niveis(k):
    c = confianca.avaliar(mases(k), cfg)
    if c.ic_superior <= 1:
        return 'Abaixo de 1'
    return 'Acima de 1' if c.ic_inferior > 1 else confianca.INSUFICIENTE


alternativas = pd.DataFrame({
    'sku': [k[0].removeprefix('item ') for k in backtests],
    'gran': [k[1] for k in backtests],
    'baseline': [k[2] for k in backtests],
    'pontual': tabela.pontual.to_list(),
    'A_preregistrada': tabela.status.to_list(),
    'B_ate_2_fronteiras': [regra_ampliada(k) for k in backtests],
    'C_jackknife': [regra_jackknife(k) for k in backtests],
    'D_tres_niveis': [regra_tres_niveis(k) for k in backtests],
})
for coluna in [c for c in alternativas if c[0] in 'ABCD']:
    print(coluna, alternativas[coluna].value_counts().to_dict())
    print(
        '   semanal:',
        alternativas
        .loc[alternativas.gran == 'semanal', coluna]
        .value_counts()
        .to_dict(),
    )
alternativas

A_preregistrada {'Insuficiente': 15, 'Ruim': 3, 'Média': 2}
   semanal: {'Insuficiente': 8}
B_ate_2_fronteiras {'Média': 7, 'Insuficiente': 7, 'Ruim': 4, 'Boa': 2}
   semanal: {'Insuficiente': 6, 'Boa': 1, 'Ruim': 1}
C_jackknife {'Média': 7, 'Ruim': 6, 'Insuficiente': 4, 'Boa': 3}
   semanal: {'Insuficiente': 4, 'Ruim': 3, 'Boa': 1}
D_tres_niveis {'Insuficiente': 15, 'Abaixo de 1': 4, 'Acima de 1': 1}
   semanal: {'Insuficiente': 7, 'Abaixo de 1': 1}


,sku,gran,baseline,pontual,A_preregistrada,B_ate_2_fronteiras,C_jackknife,D_tres_niveis
0,A,diaria,naive,Média,Insuficiente,Média,Média,Insuficiente
1,A,diaria,media_historica,Média,Insuficiente,Média,Média,Insuficiente
2,A,diaria,naive_sazonal,Média,Média,Média,Média,Abaixo de 1
3,A,semanal,naive,Ruim,Insuficiente,Insuficiente,Ruim,Insuficiente
4,A,semanal,media_historica,Boa,Insuficiente,Insuficiente,Insuficiente,Insuficiente
5,B,diaria,naive,Boa,Insuficiente,Insuficiente,Boa,Insuficiente
6,B,diaria,media_historica,Média,Insuficiente,Média,Média,Insuficiente
7,B,diaria,naive_sazonal,Média,Insuficiente,Média,Média,Insuficiente
8,B,semanal,naive,Média,Insuficiente,Insuficiente,Insuficiente,Insuficiente
9,B,semanal,media_historica,Boa,Insuficiente,Boa,Boa,Abaixo de 1


**Observação (PROPOSTA).** Fora de Insuficiente: A 5, B 13, C 16 e D 5 runs. Na semanal: A 0, B 2, C 4, D 1 de 8. Só as 3 versões Ruim e as 2 Média de A mantêm a faixa nas regras B e C.

**Interpretação (PROPOSTA, hipótese).** Faixas mais grossas (D) não resolvem: o IC cruza 1 em 15 runs. B e C classificam mais porque toleram mais incerteza, não porque há mais evidência.

## 5. Escala do MASE na diária: lag 1 contra lag 7

Sem alterar `metricas.py`: as duas escalas são calculadas aqui, por fold, só com o treino do fold. A coluna `lag1` confere com a escala de `rodar_backtest` (`assert` na célula).

In [10]:
def escala_lag(treino, m):
    treino = np.asarray(treino, dtype=float)
    return float(np.mean(np.abs(treino[m:] - treino[:-m])))


linhas, razoes = [], {}
for (sku, gran, alg), bt in backtests.items():
    if gran != 'diaria':
        continue
    serie = series[sku, gran]
    p = particoes[sku, gran]
    m1, m7, rz = [], [], []
    for f, r in zip(p.folds, bt.folds, strict=True):
        treino = serie[
            pd.Timestamp(f.treino_inicio) : pd.Timestamp(f.treino_fim)
        ]
        e1, e7 = escala_lag(treino, 1), escala_lag(treino, 7)
        assert np.isclose(e1, r.escala_mase)
        m1.append(r.metricas['mae'] / e1)
        m7.append(r.metricas['mae'] / e7)
        rz.append(e7 / e1)
    razoes[sku] = rz
    c1, c7 = confianca.avaliar(m1, cfg), confianca.avaliar(m7, cfg)
    linhas.append({
        'sku': sku.removeprefix('item '),
        'baseline': alg,
        'mase_lag1': c1.mase,
        'dp_lag1': c1.dp,
        'pontual_lag1': c1.faixa_pontual,
        'status_lag1': c1.status,
        'mase_lag7': c7.mase,
        'dp_lag7': c7.dp,
        'pontual_lag7': c7.faixa_pontual,
        'status_lag7': c7.status,
    })
escalas = pd.DataFrame(linhas)
razao = pd.Series({k: float(np.median(v)) for k, v in razoes.items()})
e1_cond = int(((razao < RAZAO_MIN) | (razao > RAZAO_MAX)).sum())
e2_cond = int((escalas.pontual_lag1 != escalas.pontual_lag7).sum())
print('razão mediana lag7/lag1 por SKU:', razao.round(3).to_dict())
print('E1: SKUs fora de [0,90; 1,10] =', e1_cond, '(recomenda se >= 2)')
print('E2: runs com faixa pontual diferente =', e2_cond, '(recomenda se >= 3)')
print(
    'status distintos em lag7:',
    escalas.status_lag7.value_counts().to_dict(),
    '| lag1:',
    escalas.status_lag1.value_counts().to_dict(),
)
escalas

razão mediana lag7/lag1 por SKU: {'item A': 0.931, 'item B': 1.001, 'item C': 0.791, 'item D': 1.045}
E1: SKUs fora de [0,90; 1,10] = 1 (recomenda se >= 2)
E2: runs com faixa pontual diferente = 1 (recomenda se >= 3)
status distintos em lag7: {'Insuficiente': 7, 'Ruim': 4, 'Média': 1} | lag1: {'Insuficiente': 7, 'Ruim': 3, 'Média': 2}


,sku,baseline,mase_lag1,dp_lag1,pontual_lag1,status_lag1,mase_lag7,dp_lag7,pontual_lag7,status_lag7
0,A,naive,0.892,0.414,Média,Insuficiente,0.968,0.439,Média,Insuficiente
1,A,media_historica,0.849,0.248,Média,Insuficiente,0.921,0.257,Média,Insuficiente
2,A,naive_sazonal,0.811,0.218,Média,Média,0.887,0.254,Média,Insuficiente
3,B,naive,0.744,0.358,Boa,Insuficiente,0.727,0.317,Boa,Insuficiente
4,B,media_historica,0.879,0.252,Média,Insuficiente,0.860,0.195,Média,Insuficiente
5,B,naive_sazonal,0.877,0.240,Média,Insuficiente,0.859,0.190,Média,Insuficiente
6,C,naive,0.836,0.268,Média,Insuficiente,1.096,0.451,Ruim,Ruim
7,C,media_historica,1.005,0.192,Ruim,Ruim,1.301,0.322,Ruim,Ruim
8,C,naive_sazonal,1.061,0.377,Ruim,Ruim,1.391,0.553,Ruim,Ruim
9,D,naive,0.787,0.198,Média,Média,0.756,0.179,Média,Média


**Observação (PROPOSTA).** Razão mediana lag 7/lag 1 por SKU: A 0,931, B 1,001, C 0,791 e D 1,045. E1: 1 SKU fora de [0,90; 1,10] (recomenda com 2 ou mais). E2: 1 run muda de faixa pontual (C naive, de Média para Ruim; recomenda com 3 ou mais). **Pela regra pré-registrada, a troca não é recomendada.** O número de Insuficiente na diária fica igual (7), com composição diferente: A naive sazonal sai de Média para Insuficiente e C naive sai de Insuficiente para Ruim.

**Interpretação (PROPOSTA, hipótese).** No item C o lag 7 reduz a escala em cerca de 21%, coerente com um padrão semanal nesse SKU; nos demais a diferença é pequena. A escala sazonal segue tendo a favor a definição para séries sazonais (Hyndman e Athanasopoulos, FPP3, seção 5.8), mas o ganho prático aqui é pequeno e a troca alteraria números aprovados em D-006.

## 6. Limitações

- 20 runs, só baselines; repetir com os candidatos do #10 e do #11, nas mesmas regras e sem usar a confiança para escolher modelo.
- 4 a 5 folds. A cobertura do IC é nominal: folds compartilham treino e a série é autocorrelacionada. A variância com 3 graus de liberdade é instável (Lohr, p. 294).
- Na semanal cada fold tem 1 ponto; na diária, 7. O ep semanal resume 4 erros absolutos.
- O MASE compara com o naive de um passo **dentro do treino**, não com o baseline no mesmo teste. MASE abaixo de 1 não significa ganhar do baseline.
- Venda é proxy de demanda; ruptura aparece como zero.
- Limites de pré-registro: as ordens de grandeza do MAE e do dp eram conhecidas ao fixar os critérios; não há carimbo externo além do hash no registro.